# Create Canada Excellence Research Chairs Awards

Creates Canada Excellence Research Chairs (CERC) awards from the program's own
site https://www.cerc.gc.ca (Tri-agency Institutional Programs Secretariat):
current chairholders (`/chairholders-titulaires/index-eng.aspx`), former
chairholders / laureates (`former-ancien-eng.aspx`) and the competition-results
page (`/results-resultats/index-eng.aspx`, Competitions 2-4). Each chair: name,
institution, agency, chair title, description; the competition number comes from
the chair's photo path (`/images/chairholders/comp{N}/...`) or the results page.
**69 chairs** (Competition 1: 19, 2: 7, 3: 9, 4: 34), 100% title/institution/PI name
(local run 2026-09-30).

**Not published per chair:** award amount, start/end dates, award number. So
`amount`, `currency` and dates are NULL (the §6.7 amount check is waived: the site
gives no per-chair amount), and `funder_scheme` names the competition.

**`funder_award_id` (§2.1.1):** synthetic `CERC-C{competition}-{given-family slug}`.
It cannot collide with the ~299 acknowledgement-derived CERC ids already in
`openalex_awards_raw` (checked 2026-09-30: none match `^CERC-C[0-9X]-[a-z]`), so these
rows do not collapse onto existing citation shells.

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320321481`
- display_name: Canada Excellence Research Chairs, Government of Canada
- ror_id / doi: from `openalex.funders.funders` (doi 10.13039/501100002784)

**Priority:** `504` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cerc_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/cerc/cerc_chairs.parquet`;

In [ ]:
%sql
SELECT source, competition, COUNT(*) as n FROM openalex.awards.cerc_raw GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
DESCRIBE openalex.awards.cerc_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.cerc_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320321481 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320321481;

## Step 2: Create CERC Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.cerc_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320321481  -- Canada Excellence Research Chairs
),

lead AS (
    -- Names were split in Python using the site's own surname span (runbook §2.4.1).
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(institution), ''),
                'country', 'CA',
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) AS lead_investigator
    FROM openalex.awards.cerc_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.chair_title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'fellowship' as funding_type,
    NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,
    'cerc_chairholders' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    CAST(NULL AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    l.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    array(l.lead_investigator) as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.cerc_raw g
JOIN lead l ON l.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): only the synthetic prefixes; unique ids.
SELECT
    assert_true(COUNT_IF(NOT funder_award_id RLIKE '^CERC-C[0-9X]-[a-z][a-z0-9-]*$') = 0,
                'funder_award_id outside the CERC-C synthetic key shape') AS shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok
FROM openalex.awards.cerc_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cerc_chairholders' AND priority = 504;

-- Insert into openalex_awards_raw with priority.
-- Priority 504: direct-from-funder ingest of the CERC program's own chairholder and
-- competition-results pages. Higher wins under the 2026-06-20 DESC dedup.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    504 as priority
FROM openalex.awards.cerc_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_cerc_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.cerc_awards;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.cerc_awards GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name AS institution, COUNT(*) as cnt FROM openalex.awards.cerc_awards GROUP BY 1 ORDER BY 2 DESC;

In [ ]:
%sql
-- Section 6.4a: one row per chairholder expected.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.cerc_awards GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.cerc_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'cerc_chairholders'
GROUP BY provenance, priority;